# ValidEval V6 — Environment and fail-closed preflight

Canonical Kaggle T4×2 notebook for the frozen V6 S1 protocol. The real smoke is always
`ENGINEERING_ONLY`; fixture and mocked runs are always `NON_EVIDENCE_FIXTURE`. Run this
notebook top-to-bottom from the tagged source package. It delegates every execution,
resume, validation, and packaging operation to tested `valideval` package code.

In [ ]:
import importlib.util
import json
import os
import subprocess
import sys
from pathlib import Path

STAGE = "environment"
MODE = os.environ.get("VALIDEVAL_EXECUTION_MODE", "fixture").strip().lower()
OUTPUT_ROOT = Path(os.environ.get("VALIDEVAL_NOTEBOOK_OUTPUT_ROOT", "kaggle_max_ceiling_outputs"))
INSTALL_SOURCE = os.environ.get("VALIDEVAL_INSTALL_SOURCE", "").strip()
if importlib.util.find_spec("valideval") is None or INSTALL_SOURCE:
    if not INSTALL_SOURCE:
        raise RuntimeError(
            "valideval is absent; set VALIDEVAL_INSTALL_SOURCE to the tagged source "
            "directory or wheel before running"
        )
    subprocess.run(
        [sys.executable, "-m", "pip", "install", INSTALL_SOURCE],
        check=True,
    )

valideval_package = __import__("valideval")
execution_package = __import__("valideval.execution", fromlist=["execution"])
__version__ = valideval_package.__version__
EXECUTION_MODES = execution_package.EXECUTION_MODES
run_notebook_stage = execution_package.run_notebook_stage

if MODE not in EXECUTION_MODES:
    raise ValueError(f"Unsupported execution mode: {MODE}")
CONFIG_BY_STAGE = {
    "environment": "configs/runs/mmlu_s1_v6.yaml",
    "mmlu": "configs/runs/mmlu_s1_v6.yaml",
    "gsm8k": "configs/runs/gsm8k_s1_v6.yaml",
    "bbh": "configs/runs/bbh_s1_v6.yaml",
}
CONFIG_PATH = os.environ.get(
    "VALIDEVAL_EXECUTION_CONFIG",
    CONFIG_BY_STAGE.get(STAGE, ""),
)
if MODE != "fixture" and STAGE in {"mmlu", "gsm8k", "bbh"}:
    os.environ["VALIDEVAL_EXECUTION_CONFIG"] = CONFIG_PATH

print(f"valideval={__version__}")
print(f"stage={STAGE} mode={MODE} config={CONFIG_PATH or 'package-set'}")

In [ ]:
RESULT = run_notebook_stage(STAGE, mode=MODE, output_root=OUTPUT_ROOT)
if MODE == "fixture":
    assert RESULT["evidence_state"] == "NON_EVIDENCE_FIXTURE"

preflight = RESULT.get("preflight", {})
print(
    json.dumps(
        {
            "status": RESULT["status"],
            "source_commit": RESULT.get("source_commit", preflight.get("source_commit")),
            "config_hash": RESULT.get("config_hash", preflight.get("config_hash")),
            "evidence_class": RESULT.get("evidence_class", RESULT.get("evidence_state")),
            "worker_status": RESULT.get("validation", RESULT.get("results")),
            "zip_path": RESULT.get("zip_path"),
        },
        indent=2,
        sort_keys=True,
    )
)

In [ ]:
EXPECTED_ZIPS = [
    OUTPUT_ROOT / "packages" / "valideval_v6_s1_mmlu_s1-v6-mmlu.zip",
    OUTPUT_ROOT / "packages" / "valideval_v6_s1_gsm8k_s1-v6-gsm8k.zip",
    OUTPUT_ROOT / "packages" / "valideval_v6_s1_bbh_s1-v6-bbh.zip",
]
print("Expected S1 ZIPs:")
for path in EXPECTED_ZIPS:
    print(path)
print("Local acceptance command:")
print("python -m valideval accept-s1 --input-dir kaggle_outputs/v6 --output-root imported/v6")
print("Runtime recalibration command:")
print(
    "python -m valideval recalibrate-runtime "
    "--input-root imported/v6 "
    "--output results/planning/runtime_recalibration_v6.json"
)